In [16]:
# Library Importing
import threading
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import os
import numpy as np
from itertools import combinations
import matplotlib.pyplot as plt
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from datetime import datetime
import warnings
import matplotlib
matplotlib.use('Agg')
import io

from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, PageBreak, Image
)
warnings.filterwarnings('ignore')

import seaborn as sns
# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [4]:
# Variable Assigning
ufa_teams = [
    "hustle", "sol", "glory", "flyers", "union", "apex", "breeze", "mechanix",
    "havoc", "alleycats", "aviators", "radicals", "windchill", "royal", "empire",
    "spiders", "phoenix", "thunderbirds", "steel", "shred", "growlers", "cascades",
    "rush", "bighorns"
]

# UPDATED: Scrape multiple years from 2021 to present
YEARS = [2021, 2022, 2023, 2024, 2025]
all_data = []

TIME_LIMIT = 300

In [ ]:
# Fetching Algorithm - UPDATED to loop through years
def scrape_all():
    options = Options()
    options.add_argument("--headless")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 10)
    try:
        for year in YEARS:
            for team in ufa_teams:
                print(f"Processing team: {team}, year: {year}")
                url = f"https://watchufa.com/stats/player-stats?team={team}&year={year}"
                driver.get(url)
                try:
                    table = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table")))
                    table_html = table.get_attribute("outerHTML")
                    tables = pd.read_html(table_html)
                    for t in tables:
                        t["Team"] = team
                        t["Year"] = year
                        all_data.append(t)
                except Exception as e:
                    print(f"Failed to get data for {team} {year}: {e}")
    finally:
        driver.quit()

print("Starting data scraping from 2021-2025...")
scrape_all()

players = pd.concat(all_data, ignore_index=True)
print(f"\nTotal rows collected: {len(players)}")
print(f"Years in dataset: {sorted(players['Year'].unique())}")
players.to_csv('data.csv')

Starting data scraping from 2021-2025...
Processing team: hustle, year: 2021
Processing team: sol, year: 2021
Processing team: glory, year: 2021
Processing team: flyers, year: 2021
Processing team: union, year: 2021
Processing team: apex, year: 2021
Failed to get data for apex 2021: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x81f3e3
	0x81f424
	0x612090
	0x65c51a
	0x65c7bb
	0x69e212
	0x67ee64
	0x69bb23
	0x67ebb6
	0x650319
	0x6510d4
	0xa746a4
	0xa6f9a9
	0xa8c940
	0x8380f8
	0x83fe4d
	0x827c18
	0x827de2
	0x8118ea
	0x744e8508
	0x7443e81c
	0x771dfcfc
	0x771dfc58
	0x772b973c

Processing team: breeze, year: 2021
Processing team: mechanix, year: 2021
Processing team: havoc, year: 2021
Failed to get data for havoc 2021: Message: 
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x81f3e3
	0x81f424
	0x612090
	0x65c51a
	0x65c7bb
	0x69e212
	0x67ee64
	0x69bb23
	0x67ebb6
	0x650319
	0x6510d4
	0xa746a4
	0xa6f9a9
	0xa8c940
	0x8380f8
	0x83fe4d
	0x827c18
	

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIG  ← edit these
# ═══════════════════════════════════════════════════════════════════════════════
TARGET    = 'OEFF'
EXCLUDE   = {'Player', 'Team', 'Year', 'POS', TARGET}
TOP_N     = 8                 # top N features used in exhaustive subset search
CV        = KFold(n_splits=5, shuffle=True, random_state=42)
PDF_OUT   = 'oeff_model_report_2021_2025.pdf'
CSV_OUT   = 'oeff_model_results_2021_2025.csv'

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════════════

def evaluate(X, y, model, cv=CV):
    """5-fold CV → (mean R², RMSE)."""
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    r2s  = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
    mses = cross_val_score(pipe, X, y, cv=cv, scoring='neg_mean_squared_error')
    return r2s.mean(), np.sqrt(-mses.mean())


def r2_color(val):
    if val >= 0.7:  return colors.HexColor('#27ae60')
    if val >= 0.4:  return colors.HexColor('#f39c12')
    return colors.HexColor('#e74c3c')


def make_bar_chart(series, title, xlabel):
    """Horizontal bar chart → PNG bytes buffer."""
    fig, ax = plt.subplots(figsize=(6.5, max(2.5, len(series) * 0.42)))
    ax.barh(series.index[::-1], series.values[::-1],
            color='#2980b9', edgecolor='white', linewidth=0.6)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=8)
    for bar, val in zip(ax.patches, series.values[::-1]):
        ax.text(val + max(series.values) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=7.5)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=140, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf


def make_scatter(y_true, y_pred, r2):
    """Actual vs Predicted scatter → PNG bytes buffer."""
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.scatter(y_true, y_pred, alpha=0.55, s=28,
               color='#2980b9', edgecolors='white', linewidth=0.4)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.2, label='Perfect fit')
    ax.set_xlabel('Actual OEFF', fontsize=9)
    ax.set_ylabel('Predicted OEFF', fontsize=9)
    ax.set_title(f'Actual vs Predicted  (R2={r2:.4f})', fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=140, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# PREPARE DATA  (uses the pre-imported `players` dataframe)
# ═══════════════════════════════════════════════════════════════════════════════
# Use the already-imported `players` dataframe directly
df = pd.read_csv('data.csv')

# Replace common null placeholders with NaN, then coerce all candidate columns to numeric
NULL_PLACEHOLDERS = ['--', '-', 'N/A', 'n/a', 'NA', 'null', 'none', 'None', '']
df.replace(NULL_PLACEHOLDERS, np.nan, inplace=True)

# Force-convert every non-excluded column to numeric (non-parseable values become NaN)
for col in df.columns:
    if col not in EXCLUDE:
        df[col] = pd.to_numeric(df[col], errors='coerce')

CANDIDATE_FEATURES = [c for c in df.columns
                      if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])]
df_clean = df[CANDIDATE_FEATURES + [TARGET]].dropna()
y = df_clean[TARGET].values
X_full = df_clean[CANDIDATE_FEATURES].values
n_rows = len(df_clean)

print(f"\nRows: {n_rows}  |  Candidate features ({len(CANDIDATE_FEATURES)}): {CANDIDATE_FEATURES}\n")


Rows: 1793  |  Candidate features (24): ['Unnamed: 0', 'G', 'PP', 'SCR', 'AST', 'GLS', 'BLK', '+/- ▼', 'Cmp', 'Cmp%', 'Y', 'TY', 'RY', 'HA', 'T', 'S', 'D', 'C', 'Hck', 'Hck%', 'Pul', 'OPP', 'DPP', 'MP']



In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# CORRELATION FILTER
# Only keep features whose |r| with OEFF meets the threshold.
# Set CORR_THRESHOLD = 0.0 to disable and keep all candidate features.
# ═══════════════════════════════════════════════════════════════════════════════
CORR_THRESHOLD = 0.2   # <- adjust as needed (e.g. 0.1, 0.2, 0.3 ...)

corr_with_target = (
    df_clean[CANDIDATE_FEATURES + [TARGET]]
    .corr()[TARGET]
    .drop(TARGET)
    .abs()
    .sort_values(ascending=False)
)

print("=" * 60)
print(f"CORRELATION FILTER  (threshold = |r| >= {CORR_THRESHOLD})")
print("=" * 60)
print(corr_with_target.to_string())

if CORR_THRESHOLD > 0.0:
    CANDIDATE_FEATURES = corr_with_target[corr_with_target >= CORR_THRESHOLD].index.tolist()
    dropped = corr_with_target[corr_with_target < CORR_THRESHOLD].index.tolist()
    print(f"\nKept  ({len(CANDIDATE_FEATURES)}): {CANDIDATE_FEATURES}")
    print(f"Dropped ({len(dropped)}): {dropped}")
    if len(CANDIDATE_FEATURES) == 0:
        raise ValueError(
            f"No features passed the correlation threshold of {CORR_THRESHOLD}. "
            "Lower CORR_THRESHOLD and try again."
        )
    df_clean = df_clean[CANDIDATE_FEATURES + [TARGET]]
    X_full = df_clean[CANDIDATE_FEATURES].values
else:
    print("\nCorrelation filter disabled. Using all candidate features.")

TOP_N = min(TOP_N, len(CANDIDATE_FEATURES))
print(f"\nFinal feature pool ({len(CANDIDATE_FEATURES)}): {CANDIDATE_FEATURES}\n")


CORRELATION FILTER  (threshold = |r| >= 0.2)
Cmp%          0.426414
+/- ▼         0.400337
HA            0.351205
SCR           0.332919
AST           0.284612
Y             0.277218
GLS           0.271644
Cmp           0.256932
TY            0.241260
OPP           0.219059
RY            0.214596
PP            0.204073
G             0.191765
MP            0.183344
Hck           0.160646
Hck%          0.157360
Unnamed: 0    0.098657
DPP           0.090278
D             0.084672
Pul           0.062780
BLK           0.061350
S             0.026688
C             0.011345
T             0.002049

Kept  (12): ['Cmp%', '+/- ▼', 'HA', 'SCR', 'AST', 'Y', 'GLS', 'Cmp', 'TY', 'OPP', 'RY', 'PP']
Dropped (12): ['G', 'MP', 'Hck', 'Hck%', 'Unnamed: 0', 'DPP', 'D', 'Pul', 'BLK', 'S', 'C', 'T']

Final feature pool (12): ['Cmp%', '+/- ▼', 'HA', 'SCR', 'AST', 'Y', 'GLS', 'Cmp', 'TY', 'OPP', 'RY', 'PP']



In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# 1. FULL-FEATURE BASELINES
# ═══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("1. FULL-FEATURE BASELINES")
print("=" * 60)

all_models = {
    'LinearRegression': LinearRegression(),
    'Ridge(a=1)':       Ridge(alpha=1),
    'Lasso(a=0.01)':    Lasso(alpha=0.01, max_iter=5000),
    'RandomForest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

baseline_results = []
for name, model in all_models.items():
    r2, rmse = evaluate(X_full, y, model)
    baseline_results.append({
        'Model': name, 'Features': 'ALL',
        'n_features': len(CANDIDATE_FEATURES),
        'CV_R2': round(r2, 4), 'CV_RMSE': round(rmse, 4)
    })
    print(f"  {name:<28} R2={r2:.4f}  RMSE={rmse:.4f}")

1. FULL-FEATURE BASELINES
  LinearRegression             R2=0.3449  RMSE=6.6490
  Ridge(a=1)                   R2=0.3450  RMSE=6.6486
  Lasso(a=0.01)                R2=0.3449  RMSE=6.6491
  RandomForest                 R2=0.4123  RMSE=6.3012
  GradientBoosting             R2=0.4332  RMSE=6.1895


In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# 2. FEATURE IMPORTANCES
# ═══════════════════════════════════════════════════════════════════════════════
print("\n2. FEATURE IMPORTANCES (Random Forest)")
rf_imp = RandomForestRegressor(n_estimators=200, random_state=42)
rf_imp.fit(StandardScaler().fit_transform(X_full), y)
importances = pd.Series(rf_imp.feature_importances_,
                        index=CANDIDATE_FEATURES).sort_values(ascending=False)
top_features = importances.head(TOP_N).index.tolist()
print(importances.to_string())
print(f"\nTop {TOP_N} for subset search: {top_features}")


2. FEATURE IMPORTANCES (Random Forest)
+/- ▼    0.238680
Cmp%     0.212231
PP       0.086040
OPP      0.080996
SCR      0.055031
TY       0.053596
RY       0.051203
AST      0.050512
HA       0.049607
Cmp      0.049341
Y        0.037767
GLS      0.034995

Top 8 for subset search: ['+/- ▼', 'Cmp%', 'PP', 'OPP', 'SCR', 'TY', 'RY', 'AST']


In [11]:
# ═══════════════════════════════════════════════════════════════════════════════
# 3. SUBSET SEARCH  (forward selection + parallel CV)
#
# SEARCH_MODE options:
#   'forward'    -- greedy forward selection (fast, recommended)
#   'exhaustive' -- try every combination up to TOP_N features (slow for TOP_N > 6)
#
# SUBSET_MAX_FEATURES caps how many features the forward search will add.
# RF_N_ESTIMATORS is reduced here for speed; raise it if you want more accuracy.
# N_JOBS controls parallelism (-1 = all CPU cores).
# ═══════════════════════════════════════════════════════════════════════════════
SEARCH_MODE        = 'forward'   # 'forward' or 'exhaustive'
SUBSET_MAX_FEATURES = TOP_N      # max features to add in forward search
RF_N_ESTIMATORS    = 50          # lower = faster; raise to 100-200 for final tuning
N_JOBS             = -1          # -1 = all CPU cores

from joblib import Parallel, delayed

search_models = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=1),
    'RandomForest':     RandomForestRegressor(n_estimators=RF_N_ESTIMATORS,
                                              random_state=42, n_jobs=N_JOBS),
}

# Redefine evaluate to support n_jobs in the CV loop
def evaluate_fast(X, y, model, cv=CV, n_jobs=N_JOBS):
    pipe = Pipeline([('scaler', StandardScaler()), ('model', model)])
    r2s  = cross_val_score(pipe, X, y, cv=cv, scoring='r2', n_jobs=n_jobs)
    mses = cross_val_score(pipe, X, y, cv=cv, scoring='neg_mean_squared_error', n_jobs=n_jobs)
    return r2s.mean(), np.sqrt(-mses.mean())

def eval_combo(feat_combo, df_clean, y, model_name, model):
    X_sub = df_clean[list(feat_combo)].values
    r2, rmse = evaluate_fast(X_sub, y, model)
    return {
        'Model': model_name, 'Features': list(feat_combo),
        'n_features': len(feat_combo),
        'CV_R2': round(r2, 4), 'CV_RMSE': round(rmse, 4)
    }

# ── Forward selection ─────────────────────────────────────────────────────────
def forward_selection(features, df_clean, y, model_name, model, max_features):
    selected, remaining = [], list(features)
    results = []
    for _ in range(min(max_features, len(features))):
        best_r2_step, best_feat = -np.inf, None
        for feat in remaining:
            candidate = selected + [feat]
            r2, rmse = evaluate_fast(df_clean[candidate].values, y, model)
            results.append({
                'Model': model_name, 'Features': candidate,
                'n_features': len(candidate),
                'CV_R2': round(r2, 4), 'CV_RMSE': round(rmse, 4)
            })
            if r2 > best_r2_step:
                best_r2_step, best_feat = r2, feat
        if best_feat is None:
            break
        selected.append(best_feat)
        remaining.remove(best_feat)
    return results

print(f"\n3. SUBSET SEARCH  mode={SEARCH_MODE}")
subset_results = []

if SEARCH_MODE == 'forward':
    # Run forward selection for each model in parallel
    all_fwd = Parallel(n_jobs=N_JOBS)(
        delayed(forward_selection)(
            top_features, df_clean, y, mname, model, SUBSET_MAX_FEATURES
        )
        for mname, model in search_models.items()
    )
    for model_results in all_fwd:
        subset_results.extend(model_results)
    print(f"  Forward selection complete ({len(subset_results)} evaluations)")

else:  # exhaustive
    all_combos = [
        (feat_combo, mname, model)
        for size in range(2, TOP_N + 1)
        for feat_combo in combinations(top_features, size)
        for mname, model in search_models.items()
    ]
    print(f"  Exhaustive: {len(all_combos)} combinations to evaluate ...")
    subset_results = Parallel(n_jobs=N_JOBS)(
        delayed(eval_combo)(fc, df_clean, y, mn, mo)
        for fc, mn, mo in all_combos
    )
    print("  Exhaustive search complete")

results_df   = pd.DataFrame(subset_results + baseline_results)
# Best result must come from subset_results only — baseline rows have Features='ALL'
# which is not a valid column list for the final model fit.
subset_df    = pd.DataFrame(subset_results)
best_result  = subset_df.sort_values('CV_R2', ascending=False).iloc[0].to_dict()
best_r2      = best_result['CV_R2']
top10 = results_df.sort_values('CV_R2', ascending=False).head(10).reset_index(drop=True)
print(f"  Best R2: {best_r2:.4f}  |  Model: {best_result['Model']}  |  Features: {best_result['Features']}")



3. SUBSET SEARCH  mode=forward
  Forward selection complete (108 evaluations)
  Best R2: 0.3856  |  Model: RandomForest  |  Features: ['+/- ▼', 'Cmp%', 'OPP', 'SCR', 'PP', 'RY', 'AST']


In [12]:
# ═══════════════════════════════════════════════════════════════════════════════
# 4. FINAL MODEL FIT
# ═══════════════════════════════════════════════════════════════════════════════
best_feats  = best_result['Features']
X_best      = df_clean[best_feats].values
X_best_sc   = StandardScaler().fit_transform(X_best)

if best_result['Model'] == 'RandomForest':
    final_model = RandomForestRegressor(n_estimators=200, random_state=42)
    final_model.fit(X_best_sc, y)
    coef_series = pd.Series(final_model.feature_importances_,
                            index=best_feats).sort_values(ascending=False)
    coef_label  = 'Feature Importance'
    intercept   = None
else:
    final_model = (LinearRegression() if best_result['Model'] == 'LinearRegression'
                   else Ridge(alpha=1))
    final_model.fit(X_best_sc, y)
    coef_series = pd.Series(final_model.coef_,
                            index=best_feats).sort_values(ascending=False)
    coef_label  = 'Standardised Coefficient'
    intercept   = final_model.intercept_

y_pred        = final_model.predict(X_best_sc)
insample_r2   = r2_score(y, y_pred)
insample_rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"\nBest: {best_result['Model']}  features={best_feats}")
print(f"CV R2={best_r2:.4f}  In-sample R2={insample_r2:.4f}  RMSE={insample_rmse:.4f}")


Best: RandomForest  features=['+/- ▼', 'Cmp%', 'OPP', 'SCR', 'PP', 'RY', 'AST']
CV R2=0.3856  In-sample R2=0.9183  RMSE=2.3471


In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# 5. SAVE CSV
# ═══════════════════════════════════════════════════════════════════════════════
results_df.sort_values('CV_R2', ascending=False).to_csv(CSV_OUT, index=False)
print(f"CSV saved: {CSV_OUT}")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. BUILD PDF
# ═══════════════════════════════════════════════════════════════════════════════
print("Building PDF ...")

styles = getSampleStyleSheet()

def sty(base, **kw):
    return ParagraphStyle(f'_custom_{base}', parent=styles.get(base, styles['Normal']), **kw)

s_title    = sty('Normal', fontName='Helvetica-Bold', fontSize=22,
                 textColor=colors.HexColor('#1a2c47'), alignment=TA_CENTER, spaceAfter=4)
s_subtitle = sty('Normal', fontName='Helvetica', fontSize=11,
                 textColor=colors.HexColor('#5d6d7e'), alignment=TA_CENTER, spaceAfter=2)
s_h1       = sty('Normal', fontName='Helvetica-Bold', fontSize=14,
                 textColor=colors.HexColor('#1a2c47'), spaceBefore=14, spaceAfter=6)
s_h2       = sty('Normal', fontName='Helvetica-Bold', fontSize=11,
                 textColor=colors.HexColor('#2980b9'), spaceBefore=10, spaceAfter=4)
s_body     = sty('Normal', fontName='Helvetica', fontSize=9, leading=13, spaceAfter=4)
s_bold     = sty('Normal', fontName='Helvetica-Bold', fontSize=9, spaceAfter=4)

DIVIDER   = colors.HexColor('#c8d6e5')
HDR_DARK  = colors.HexColor('#1a2c47')
HDR_BLUE  = colors.HexColor('#2980b9')
ROW_ALT   = colors.HexColor('#f0f4f8')

def base_tbl_style(hdr_color=HDR_DARK):
    return TableStyle([
        ('BACKGROUND',     (0,0), (-1,0),  hdr_color),
        ('TEXTCOLOR',      (0,0), (-1,0),  colors.white),
        ('FONTNAME',       (0,0), (-1,0),  'Helvetica-Bold'),
        ('FONTSIZE',       (0,0), (-1,-1), 8.5),
        ('ALIGN',          (0,0), (-1,-1), 'CENTER'),
        ('VALIGN',         (0,0), (-1,-1), 'MIDDLE'),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, ROW_ALT]),
        ('GRID',           (0,0), (-1,-1), 0.4, DIVIDER),
        ('TOPPADDING',     (0,0), (-1,-1), 4),
        ('BOTTOMPADDING',  (0,0), (-1,-1), 4),
        ('LEFTPADDING',    (0,0), (-1,-1), 6),
        ('RIGHTPADDING',   (0,0), (-1,-1), 6),
    ])

doc   = SimpleDocTemplate(PDF_OUT, pagesize=letter,
                          leftMargin=0.75*inch, rightMargin=0.75*inch,
                          topMargin=0.75*inch,  bottomMargin=0.75*inch)
story = []
W     = 7.0 * inch

# ── Cover ─────────────────────────────────────────────────────────────────────
story.append(Spacer(1, 0.6*inch))
story.append(Paragraph("Ultimate Frisbee", s_subtitle))
story.append(Paragraph("OEFF Model Selection Report", s_title))
story.append(Paragraph("2021-2025 Data", s_subtitle))
story.append(HRFlowable(width=W, thickness=2, color=HDR_BLUE, spaceAfter=10))
story.append(Paragraph(f"Generated: {datetime.now().strftime('%B %d, %Y  %H:%M')}", s_subtitle))
story.append(Spacer(1, 0.25*inch))

corr_label = f'>= {CORR_THRESHOLD}' if CORR_THRESHOLD > 0 else 'Off'
summary = Table(
    [['Dataset Rows', 'Features After Filter', 'Target', 'Corr Threshold', 'Years'],
     [str(n_rows), str(len(CANDIDATE_FEATURES)), TARGET, corr_label, '2021-2025']],
    colWidths=[W/5]*5
)
summary.setStyle(base_tbl_style(HDR_BLUE))
story.append(summary)
story.append(PageBreak())


# ── Section 0: Correlation Filter ────────────────────────────────────────────
story.append(Paragraph("0. Correlation Filter Results", s_h1))
story.append(HRFlowable(width=W, thickness=1, color=DIVIDER, spaceAfter=6))
_thresh_desc = (f"Features with |r| >= {CORR_THRESHOLD} were kept."
                if CORR_THRESHOLD > 0 else "Correlation filter was disabled (threshold = 0).")
story.append(Paragraph(_thresh_desc, s_body))

corr_data = [['Feature', '|Corr with OEFF|', 'Included']]
for feat, val in corr_with_target.items():
    included = 'Yes' if feat in CANDIDATE_FEATURES else 'No'
    corr_data.append([feat, f'{val:.4f}', included])
corr_tbl = Table(corr_data, colWidths=[2.5*inch, 2.5*inch, 2.0*inch])
corr_tbl.setStyle(base_tbl_style())
for i, feat in enumerate(corr_with_target.index, 1):
    if feat not in CANDIDATE_FEATURES:
        corr_tbl.setStyle(TableStyle([
            ('TEXTCOLOR', (0,i), (-1,i), colors.HexColor('#aaaaaa')),
            ('TEXTCOLOR', (2,i), (2,i),  colors.HexColor('#e74c3c')),
            ('FONTNAME',  (2,i), (2,i),  'Helvetica-Bold'),
        ]))
    else:
        corr_tbl.setStyle(TableStyle([
            ('TEXTCOLOR', (2,i), (2,i),  colors.HexColor('#27ae60')),
            ('FONTNAME',  (2,i), (2,i),  'Helvetica-Bold'),
        ]))
story.append(corr_tbl)
story.append(Spacer(1, 0.15*inch))
story.append(Image(
    make_bar_chart(corr_with_target.sort_values(), 'Absolute Correlation with OEFF', '|r|'),
    width=W, height=max(2.5*inch, len(corr_with_target)*0.3*inch)))
story.append(PageBreak())

# ── Section 1: Baselines ──────────────────────────────────────────────────────
story.append(Paragraph("1. Full-Feature Baseline Models", s_h1))
story.append(HRFlowable(width=W, thickness=1, color=DIVIDER, spaceAfter=6))
story.append(Paragraph(
    "Each model trained on all candidate features, evaluated with 5-fold cross-validation.",
    s_body))

bl_data = [['Model', 'Num Features', 'CV R2', 'CV RMSE']]
for r in sorted(baseline_results, key=lambda x: x['CV_R2'], reverse=True):
    bl_data.append([r['Model'], str(r['n_features']),
                    f"{r['CV_R2']:.4f}", f"{r['CV_RMSE']:.4f}"])
bl_tbl = Table(bl_data, colWidths=[2.6*inch, 1.4*inch, 1.5*inch, 1.5*inch])
bl_tbl.setStyle(base_tbl_style())
for i, r in enumerate(sorted(baseline_results, key=lambda x: x['CV_R2'], reverse=True), 1):
    bl_tbl.setStyle(TableStyle([
        ('TEXTCOLOR', (2,i), (2,i), r2_color(r['CV_R2'])),
        ('FONTNAME',  (2,i), (2,i), 'Helvetica-Bold'),
    ]))
story.append(bl_tbl)

bl_series = pd.Series({r['Model']: r['CV_R2'] for r in baseline_results}).sort_values()
story.append(Spacer(1, 0.15*inch))
story.append(Image(make_bar_chart(bl_series, 'Baseline Model CV R2', 'CV R2'),
                   width=W, height=2.8*inch))
story.append(PageBreak())

# ── Section 2: Feature Importances ───────────────────────────────────────────
story.append(Paragraph("2. Feature Importances (Random Forest)", s_h1))
story.append(HRFlowable(width=W, thickness=1, color=DIVIDER, spaceAfter=6))
story.append(Paragraph(
    f"A Random Forest on all features was used to rank predictors. "
    f"The top {TOP_N} were selected for the exhaustive subset search.", s_body))

imp_data = [['Feature', 'Importance', 'In Subset Search']]
for feat, val in importances.items():
    imp_data.append([feat, f"{val:.4f}", 'Yes' if feat in top_features else ''])
imp_tbl = Table(imp_data, colWidths=[2.2*inch, 1.6*inch, 3.2*inch])
imp_tbl.setStyle(base_tbl_style())
for i, feat in enumerate(importances.index, 1):
    if feat in top_features:
        imp_tbl.setStyle(TableStyle([
            ('BACKGROUND', (0,i), (-1,i), colors.HexColor('#eaf4fb')),
            ('TEXTCOLOR',  (2,i), (2,i),  colors.HexColor('#27ae60')),
            ('FONTNAME',   (2,i), (2,i),  'Helvetica-Bold'),
        ]))
story.append(imp_tbl)
story.append(Spacer(1, 0.15*inch))
story.append(Image(make_bar_chart(importances, 'Feature Importances for OEFF', 'Importance'),
                   width=W, height=max(2.5*inch, len(importances)*0.3*inch)))
story.append(PageBreak())

# ── Section 3: Top 10 Subsets ─────────────────────────────────────────────────
story.append(Paragraph("3. Top 10 Feature Subset Models", s_h1))
story.append(HRFlowable(width=W, thickness=1, color=DIVIDER, spaceAfter=6))
story.append(Paragraph(
    f"Exhaustive search across all 2-to-{TOP_N} feature combinations "
    "(Linear Regression, Ridge, Random Forest), ranked by CV R2.", s_body))

t10_data = [['Rank', 'Model', 'n', 'CV R2', 'CV RMSE', 'Features']]
for i, row in top10.iterrows():
    fs = ', '.join(row['Features']) if isinstance(row['Features'], list) else str(row['Features'])
    t10_data.append([str(i+1), row['Model'], str(row['n_features']),
                     f"{row['CV_R2']:.4f}", f"{row['CV_RMSE']:.4f}", fs])
t10_tbl = Table(t10_data,
                colWidths=[0.4*inch, 1.45*inch, 0.3*inch, 0.75*inch, 0.8*inch, 3.3*inch])
t10_tbl.setStyle(base_tbl_style())
for i, row in top10.iterrows():
    t10_tbl.setStyle(TableStyle([
        ('TEXTCOLOR', (3,i+1), (3,i+1), r2_color(row['CV_R2'])),
        ('FONTNAME',  (3,i+1), (3,i+1), 'Helvetica-Bold'),
    ]))
story.append(t10_tbl)
story.append(PageBreak())

# ── Section 4: Best Model Detail ──────────────────────────────────────────────
story.append(Paragraph("4. Best Model — Full Detail", s_h1))
story.append(HRFlowable(width=W, thickness=1, color=DIVIDER, spaceAfter=6))

box = Table(
    [[Paragraph(f"<b>Best Model:</b> {best_result['Model']}", s_body),
      Paragraph(f"<b>CV R2:</b> {best_result['CV_R2']:.4f}", s_body),
      Paragraph(f"<b>CV RMSE:</b> {best_result['CV_RMSE']:.4f}", s_body)],
     [Paragraph(f"<b>In-sample R2:</b> {insample_r2:.4f}", s_body),
      Paragraph(f"<b>In-sample RMSE:</b> {insample_rmse:.4f}", s_body),
      Paragraph(f"<b>Num Features:</b> {len(best_feats)}", s_body)]],
    colWidths=[W/3]*3
)
box.setStyle(TableStyle([
    ('BACKGROUND',    (0,0), (-1,-1), colors.HexColor('#eaf4fb')),
    ('BOX',           (0,0), (-1,-1), 1.2, HDR_BLUE),
    ('INNERGRID',     (0,0), (-1,-1), 0.5, colors.HexColor('#aed6f1')),
    ('TOPPADDING',    (0,0), (-1,-1), 6),
    ('BOTTOMPADDING', (0,0), (-1,-1), 6),
]))
story.append(box)
story.append(Spacer(1, 0.1*inch))
story.append(Paragraph("<b>Features used:</b> " + ", ".join(best_feats), s_body))
if intercept is not None:
    story.append(Paragraph(f"<b>Intercept:</b> {intercept:.4f}", s_body))

story.append(Spacer(1, 0.08*inch))
story.append(Paragraph(coef_label, s_h2))

c_data = [[coef_label, 'Value']] + [[f, f"{v:.4f}"] for f, v in coef_series.items()]
c_tbl  = Table(c_data, colWidths=[3.5*inch, 3.5*inch])
c_tbl.setStyle(base_tbl_style(HDR_BLUE))
story.append(c_tbl)

story.append(Spacer(1, 0.15*inch))
story.append(Image(make_bar_chart(coef_series, f'{coef_label} - Best Model', coef_label),
                   width=W*0.75, height=2.6*inch))

story.append(Spacer(1, 0.15*inch))
story.append(Paragraph("Actual vs Predicted (in-sample)", s_h2))
story.append(Image(make_scatter(y, y_pred, insample_r2), width=3.6*inch, height=3.2*inch))
story.append(PageBreak())

# ── Section 5: Top 30 ─────────────────────────────────────────────────────────
story.append(Paragraph("5. Full Results — Top 30 Combinations", s_h1))
story.append(HRFlowable(width=W, thickness=1, color=DIVIDER, spaceAfter=6))
story.append(Paragraph(
    f"Complete results saved to <i>{CSV_OUT}</i>. Table shows top 30 by CV R2.", s_body))

top30    = results_df.sort_values('CV_R2', ascending=False).head(30).reset_index(drop=True)
f30_data = [['#', 'Model', 'n', 'CV R2', 'CV RMSE', 'Features']]
for i, row in top30.iterrows():
    fs = ', '.join(row['Features']) if isinstance(row['Features'], list) else str(row['Features'])
    f30_data.append([str(i+1), row['Model'], str(row['n_features']),
                     f"{row['CV_R2']:.4f}", f"{row['CV_RMSE']:.4f}", fs])
f30_tbl = Table(f30_data,
                colWidths=[0.35*inch, 1.4*inch, 0.3*inch, 0.7*inch, 0.75*inch, 3.5*inch])
f30_tbl.setStyle(base_tbl_style())
story.append(f30_tbl)

# ── Build ──────────────────────────────────────────────────────────────────────
doc.build(story)
print(f"PDF report saved: {PDF_OUT}")

# Note: os.startfile() only works on Windows. Removed for cross-platform compatibility.
print("\nAnalysis complete! Check the output files:")
print(f"  - {PDF_OUT}")
print(f"  - {CSV_OUT}")


CSV saved: oeff_model_results_2021_2025.csv
Building PDF ...
PDF report saved: oeff_model_report_2021_2025.pdf

Analysis complete! Check the output files:
  - oeff_model_report_2021_2025.pdf
  - oeff_model_results_2021_2025.csv


In [14]:
os.startfile('oeff_model_report_2021_2025.pdf')

In [15]:
# Look into random tree depth (Say it is training too much to 'training data')